# Credit Risk Model: Predicting Loan Defaults

**Qais Sawar** · BSc Financial Mathematics, University of Nottingham

In my Standard Chartered credit analyst simulation I assessed one company's credit risk by hand, using its financial ratios. Banks also need to do this for thousands of borrowers at once, so they use credit scoring models.

In this notebook I build a simple credit scoring model in Python using the German Credit dataset: 1,000 real loans from a German bank, each labelled as repaid (good) or defaulted (bad).

1. How default rates differ between types of borrower
2. A logistic regression model that gives each borrower a probability of default (PD)
3. How well the model separates good and bad borrowers (AUC and Gini)
4. Score bands: whether higher-risk bands really default more
5. Expected loss on the loan portfolio (PD × LGD × EAD)
6. Choosing an approval cut-off when a bad loan costs more than a rejected good one
7. Limitations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

## 1. Load the data

The dataset comes from the UCI Machine Learning Repository (Statlog German Credit Data) and is downloaded from OpenML. Each row is one loan. Amounts are in Deutsche Marks (DM), Germany's currency before the euro.

I create a `default` column: 1 if the loan was bad, 0 if it was repaid.

In [ ]:
data = fetch_openml("credit-g", version=1, as_frame=True)
df = data.frame.copy()
df["default"] = (df["class"] == "bad").astype(int)
df = df.drop(columns="class")

print(f"Loans: {len(df)}  |  Defaults: {df['default'].sum()}  |  Default rate: {100 * df['default'].mean():.1f}%")
df.head()

## 2. Which borrowers default more?

Before building a model, I look at the default rate for different groups of borrowers. If the default rate changes a lot between groups, that feature is likely to be useful for predicting defaults.

In [ ]:
def default_rates(col):
    t = df.groupby(col, observed=True)["default"].agg(loans="count", default_rate="mean")
    t["default_rate"] = (100 * t["default_rate"]).round(1)
    return t.sort_values("default_rate", ascending=False)

df["duration_band"] = pd.cut(df["duration"], [0, 12, 24, 36, 100], labels=["Up to 1 year", "1-2 years", "2-3 years", "Over 3 years"])
df["age_band"] = pd.cut(df["age"], [0, 25, 35, 50, 120], labels=["Under 25", "25-35", "35-50", "Over 50"])

for col in ["checking_status", "credit_history", "savings_status", "duration_band", "age_band"]:
    print(f"\n--- {col} ---")
    print(default_rates(col).to_string())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
t = default_rates("checking_status")
ax1.barh(t.index.astype(str), t["default_rate"], color="#1f2a44")
ax1.axvline(100 * df["default"].mean(), color="red", ls="--", label="Average")
ax1.set_title("Default rate by current account balance (DM)"); ax1.set_xlabel("%"); ax1.legend()
t = default_rates("duration_band").sort_index()
ax2.bar(t.index.astype(str), t["default_rate"], color="#1f2a44")
ax2.axhline(100 * df["default"].mean(), color="red", ls="--")
ax2.set_title("Default rate by loan length"); ax2.set_ylabel("%")
plt.tight_layout(); plt.show()

## 3. Building the model

I use **logistic regression**, a common model for credit scoring because it is simple to explain: each feature pushes the probability of default up or down. The model outputs a **probability of default (PD)** between 0 and 1 for each borrower.

- I train the model on 70% of the loans and test it on the other 30%, which it has never seen. Testing on the training data would make it look better than it is.
- I leave out **personal status (which includes sex) and foreign worker status**. In the UK, lenders can't base decisions on protected characteristics like these, and using them would be unfair even if they improved the model.
- Text categories (like checking account status) are turned into 0/1 columns, and numbers are put on the same scale.

In [ ]:
excluded = ["personal_status", "foreign_worker", "duration_band", "age_band", "default"]
X = df.drop(columns=excluded)
y = df["default"]

categorical = X.select_dtypes(exclude="number").columns.tolist()
numeric = X.select_dtypes(include="number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", StandardScaler(), numeric)])),
    ("logit", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)

pd_test = model.predict_proba(X_test)[:, 1]
print(f"Training loans: {len(X_train)}  |  Test loans: {len(X_test)}")
print(f"Average predicted PD on test loans: {100 * pd_test.mean():.1f}%  |  Actual default rate: {100 * y_test.mean():.1f}%")

## 4. How good is the model?

The **AUC** is the chance that, if I pick one borrower who defaulted and one who didn't, the model gives the defaulter the higher PD. 0.5 is a coin flip and 1 is perfect. Banks often quote the **Gini coefficient** instead, which is 2 × AUC − 1.

To check the model adds something, I compare it with a very simple model that only uses the current account balance.

In [ ]:
auc = roc_auc_score(y_test, pd_test)

simple = Pipeline([("prep", OneHotEncoder(handle_unknown="ignore")), ("logit", LogisticRegression(max_iter=1000))])
simple.fit(X_train[["checking_status"]], y_train)
pd_simple = simple.predict_proba(X_test[["checking_status"]])[:, 1]
auc_simple = roc_auc_score(y_test, pd_simple)

print(f"Full model:            AUC {auc:.3f}  |  Gini {2 * auc - 1:.3f}")
print(f"Current account only:  AUC {auc_simple:.3f}  |  Gini {2 * auc_simple - 1:.3f}")

for p, label, c in [(pd_test, "Full model", "#1f2a44"), (pd_simple, "Current account only", "#9e9e9e")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, color=c, label=f"{label} (AUC {roc_auc_score(y_test, p):.2f})")
plt.plot([0, 1], [0, 1], color="red", ls="--", label="Random guess")
plt.xlabel("Share of good borrowers wrongly flagged"); plt.ylabel("Share of defaulters caught")
plt.title("ROC curve (test loans)"); plt.legend(); plt.show()

### What drives the PD?

Each coefficient shows how a feature changes the odds of default. Positive means riskier, negative means safer. These are the strongest in each direction.

In [ ]:
names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["logit"].coef_[0], index=names)
coefs.index = coefs.index.str.replace("cat__", "").str.replace("num__", "")
drivers = pd.concat([coefs.nlargest(6), coefs.nsmallest(6)]).sort_values()

plt.figure(figsize=(9, 5))
plt.barh(drivers.index, drivers.values, color=["#2e7d32" if v < 0 else "#c62828" for v in drivers.values])
plt.title("Strongest drivers of default (red = riskier, green = safer)"); plt.xlabel("Coefficient")
plt.tight_layout(); plt.show()

## 5. Score bands

Banks usually group borrowers into risk bands rather than using the raw PD. I split the test loans into 5 equal-sized bands by PD. If the model works, each band should have a higher **actual** default rate than the one before it.

In [ ]:
test = X_test.copy()
test["default"] = y_test.values
test["PD"] = pd_test
test["band"] = pd.qcut(test["PD"], 5, labels=["A (safest)", "B", "C", "D", "E (riskiest)"])

bands = test.groupby("band", observed=True).agg(
    loans=("PD", "count"),
    average_PD=("PD", "mean"),
    actual_default_rate=("default", "mean"))
bands[["average_PD", "actual_default_rate"]] *= 100
print(bands.round(1).to_string())

x = np.arange(len(bands))
plt.bar(x - 0.2, bands["average_PD"], 0.4, label="Predicted PD", color="#9e9e9e")
plt.bar(x + 0.2, bands["actual_default_rate"], 0.4, label="Actual default rate", color="#1f2a44")
plt.xticks(x, bands.index); plt.ylabel("%"); plt.title("Predicted vs actual default rate by band"); plt.legend()
plt.show()

## 6. Expected loss

The PD alone doesn't say how much money is at risk. Under the Basel rules, banks estimate **expected loss** as:

$$\text{EL} = \text{PD} \times \text{LGD} \times \text{EAD}$$

- **PD**: probability of default, from the model
- **LGD**: loss given default, the share of the loan lost if the borrower defaults. The dataset doesn't have this, so I assume **45%**, the figure Basel's foundation approach uses for senior unsecured loans
- **EAD**: exposure at default, the amount owed. I use the loan amount

In [ ]:
LGD = 0.45
test["EAD"] = test["credit_amount"]
test["EL"] = test["PD"] * LGD * test["EAD"]

print(f"Test portfolio: {len(test)} loans, total exposure {test['EAD'].sum():,.0f} DM")
print(f"Expected loss: {test['EL'].sum():,.0f} DM  ({100 * test['EL'].sum() / test['EAD'].sum():.1f}% of exposure)")

el = test.groupby("band", observed=True).agg(exposure=("EAD", "sum"), expected_loss=("EL", "sum"))
el["share_of_exposure_%"] = 100 * el["exposure"] / el["exposure"].sum()
el["share_of_expected_loss_%"] = 100 * el["expected_loss"] / el["expected_loss"].sum()
el.round(0)

## 7. Choosing an approval cut-off

A bank has to decide which loans to approve. If it approves anyone with a PD below a cut-off, a lower cut-off means fewer defaults but also turning away more good customers.

The dataset's documentation says that approving a borrower who defaults is **5 times as costly** as rejecting a good borrower. I use that to find the cut-off with the lowest total cost.

In [ ]:
rows = []
for cut in np.arange(0.10, 0.75, 0.05):
    approved = test["PD"] < cut
    bad_approved = (approved & (test["default"] == 1)).sum()
    good_rejected = (~approved & (test["default"] == 0)).sum()
    rows.append({"PD cut-off": round(cut, 2),
                 "Approval rate (%)": 100 * approved.mean(),
                 "Default rate of approved (%)": 100 * test.loc[approved, "default"].mean() if approved.any() else np.nan,
                 "Defaulters rejected (%)": 100 * (~approved & (test["default"] == 1)).sum() / test["default"].sum(),
                 "Cost (5 x bad approved + good rejected)": 5 * bad_approved + good_rejected})
cutoffs = pd.DataFrame(rows).round({"PD cut-off": 2, "Approval rate (%)": 1, "Default rate of approved (%)": 1, "Defaulters rejected (%)": 1})
best = cutoffs.loc[cutoffs["Cost (5 x bad approved + good rejected)"].idxmin()]
print(f"Lowest-cost cut-off: approve if PD < {best['PD cut-off']:.2f}  "
      f"(approves {best['Approval rate (%)']:.0f}% of applicants, {best['Default rate of approved (%)']:.0f}% of whom default)")
cutoffs

## 8. Limitations

- **Old and small data:** 1,000 loans from one German bank, collected decades ago. Real models use hundreds of thousands of recent loans.
- **Only approved loans:** the bank only knows whether loans it *approved* defaulted. People it rejected never got a loan, so the model learns from a biased sample.
- **Assumed LGD:** the 45% LGD is a regulatory assumption, not measured from this data. Real LGD depends on collateral and recovery.
- **No economic conditions:** default rates rise in recessions, but this model has no information about the economy, so the PDs don't change with the cycle.
- **One test split:** with only 300 test loans, the AUC would move a bit with a different random split.
- **Fairness:** I left out sex and foreign worker status, but other features (like age or job) can still be linked to protected characteristics, which real lenders have to check.

## 9. My results

(Write this after running the notebook: what the AUC and Gini were, whether the bands ranked correctly, what the expected loss was, which cut-off had the lowest cost, and what surprised you.)